## Filesystem MCP Server

In [ ]:
from mcp.server.fastmcp import FastMCP
from pathlib import Path
import subprocess

mcp = FastMCP("filesystem-cli")

ROOT = Path.home()


def safe_path(path: str = ".") -> Path:
    requested = (ROOT / path).resolve()
    root = ROOT.resolve()

    if not str(requested).startswith(str(root)):
        raise ValueError(f"Path outside allowed root: {path}")

    return requested


def run_cmd(cmd: list[str]) -> str:
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=10,
        shell=False,
    )
    return result.stdout.strip() or result.stderr.strip()


@mcp.resource("fs://")
def list_root():
    return [
        {
            "name": item.name,
            "path": str(item.relative_to(ROOT)),
            "type": "directory" if item.is_dir() else "file",
        }
        for item in ROOT.iterdir()
    ]


@mcp.resource("fs://{path}")
def read_fs_resource(path: str):
    p = safe_path(path)

    if p.is_dir():
        return [
            {
                "name": item.name,
                "path": str(item.relative_to(ROOT)),
                "type": "directory" if item.is_dir() else "file",
            }
            for item in p.iterdir()
        ]

    if p.is_file():
        return p.read_text(errors="replace")

    raise FileNotFoundError(path)


@mcp.tool()
def pwd() -> str:
    return str(ROOT)


@mcp.tool()
def ls(path: str = ".") -> str:
    p = safe_path(path)
    return run_cmd(["ls", "-la", str(p)])


@mcp.tool()
def cat(path: str) -> str:
    p = safe_path(path)

    if not p.is_file():
        raise ValueError(f"Not a file: {path}")

    return p.read_text(errors="replace")


@mcp.tool()
def grep(path: str, pattern: str, max_lines: int = 50) -> str:
    p = safe_path(path)

    if not p.is_file():
        raise ValueError(f"Not a file: {path}")

    result = subprocess.run(
        ["grep", "-n", pattern, str(p)],
        capture_output=True,
        text=True,
        timeout=10,
        shell=False,
    )

    lines = result.stdout.splitlines()[:max_lines]
    return "\n".join(lines) if lines else "Keine Treffer."


@mcp.tool()
def df() -> str:
    return run_cmd(["df", "-h"])


@mcp.tool()
def uname() -> str:
    return run_cmd(["uname", "-a"])

In [ ]:
await mcp.run_sse_async()